# 🏗️ Python OOP & Class Patterns for LC — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A class is a blueprint. An instance is the house built from it. `__init__` is the construction crew. Dunder methods (`__lt__`, `__iter__`) are contracts — Python checks for them when you do `<`, `for`, `with`. Stateful classes in LC are like machines: `__init__` wires the machine, methods operate it, and the heap or deque inside is the hidden engine.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Visual Model — Class Anatomy](#1) |
| 2 | [Core Dunder Methods](#2) |
| 3 | [Decision Map — Which Dunder When](#3) |
| 4 | [Pattern 1: ListNode — the LC linked list node](#4) |
| 5 | [Pattern 2: __lt__ for heap ordering](#5) |
| 6 | [Pattern 3: __iter__ / __next__ — iterator protocol](#6) |
| 7 | [Pattern 4: @property, @staticmethod, @classmethod](#7) |
| 8 | [Pattern 5: MinStack — stateful class (LC 155)](#8) |
| 9 | [Pattern 6: MedianFinder — two-heap design (LC 295)](#9) |
| 10 | [Pattern 7: LRUCache skeleton (LC 146)](#10) |
| 11 | [Pattern 8: KthLargest (LC 703)](#11) |
| 12 | [Full Decision Map](#12) |
| 13 | [Interview Cheat Sheet](#13) |


<a id='1'></a>

## 1. Visual Model — Class Anatomy

```
CLASS BLUEPRINT                        INSTANCE (the house)
───────────────────────────────────────────────────────────────
class MinStack:                        s = MinStack()
    def __init__(self):                ← runs once at construction
        self.stack = []                ← instance variable: s.stack
        self.min_stack = []            ← instance variable: s.min_stack

    def push(self, val):               s.push(3)
        ...

CLASS VARIABLE vs INSTANCE VARIABLE:
  class Counter:
      count = 0                        ← shared across ALL instances
      def __init__(self):
          self.value = 0               ← unique to THIS instance

DUNDER PROTOCOL — Python calls these automatically:
  __init__    → called at MyClass()
  __repr__    → called by repr(obj) and in REPL
  __str__     → called by print(obj) and str(obj)
  __lt__      → called when obj < other  (heap uses this)
  __eq__      → called when obj == other
  __len__     → called by len(obj)
  __iter__    → called by for x in obj:
  __next__    → called by next(obj) inside a for loop
  __enter__   → called by with obj as ...:
  __exit__    → called at end of with block

HEAP ORDERING RULE:
  heapq pushes/pops using < comparisons.
  If you push objects, define __lt__.
  OR wrap in tuple: heapq.heappush(h, (priority, obj))
```


<a id='2'></a>

## 2. Core Dunder Methods

```
DUNDER          TRIGGERED BY          RETURNS          USE CASE
──────────────────────────────────────────────────────────────────────
__init__        MyClass(args)         None             construction
__repr__        repr(obj)             str              debug repr
__str__         print(obj), str(obj)  str              human display
__eq__          obj == other          bool             equality check
__lt__          obj < other           bool             heap ordering
__le__          obj <= other          bool             sorting
__hash__        hash(obj)             int              dict key / set member
__len__         len(obj)              int              container size
__getitem__     obj[key]              any              indexing
__setitem__     obj[key] = val        None             index assignment
__contains__    x in obj              bool             membership test
__iter__        for x in obj          iterator         iteration
__next__        next(iterator)        item             next item
__enter__       with obj as x:        obj/resource     context setup
__exit__        end of with block     bool             cleanup/suppress exc
__call__        obj()                 any              callable objects

NOTE: __repr__ should be unambiguous (eval-able). __str__ should be readable.
      If only __repr__ is defined, str() falls back to repr().
```


<a id='3'></a>

## 3. Decision Map — Which Dunder When

```
NEED TO                                    DEFINE
──────────────────────────────────────────────────────────────────
Push custom objects onto heap              __lt__(self, other)
Use object as dict key or set member       __eq__ + __hash__
Make object iterable (for x in obj)       __iter__ returning self + __next__
Make object usable in with statement       __enter__ + __exit__
Make object callable like a function       __call__
Show useful debug output in REPL           __repr__
Show user-friendly print output            __str__
Support len(obj)                           __len__
Support obj[index]                         __getitem__

HEAP ORDERING SHORT RULES:
  Option A: define __lt__ on the class
  Option B: push tuples (priority, object) — tuple comparison uses first element
  Option C: push (priority, tie_breaker, object) — tie_breaker breaks __lt__ ambiguity

LC STATEFUL CLASS PATTERN:
  LC 155 MinStack:   __init__ sets up storage; push/pop/top/getMin operate it
  LC 295 MedianFinder: __init__ sets two heaps; addNum/findMedian use them
  LC 146 LRUCache:   __init__ sets capacity + OrderedDict; get/put maintain LRU order
```


<a id='4'></a>

## 4. 🧩 Pattern 1: ListNode — the Standard LC Linked List Node

---

```
PROBLEM:  Nearly every linked list LC problem uses ListNode. Know it cold.

DEFINITION:
  class ListNode:
      def __init__(self, val=0, next=None):
          self.val  = val
          self.next = next    ← points to next node, None if tail

BUILDING A LIST from [1, 2, 3]:
  node1 = ListNode(1)
  node2 = ListNode(2)
  node3 = ListNode(3)
  node1.next = node2
  node2.next = node3
  → 1 → 2 → 3 → None

TRAVERSAL:
  curr = head
  while curr:
      print(curr.val)
      curr = curr.next

DUMMY HEAD PATTERN:
  dummy = ListNode(0)          ← sentinel avoids edge cases on head removal
  dummy.next = head
  curr = dummy
  ... manipulate ...
  return dummy.next

KEY INSIGHT: __repr__ on ListNode lets you print the whole chain without
             writing a traversal every time — save it in your template.
```


In [ ]:
class ListNode:
    """Standard LC linked list node — memorize this definition."""
    def __init__(self, val=0, next=None):
        self.val  = val
        self.next = next

    def __repr__(self) -> str:
        # __repr__: lets print(node) show the full chain instead of <object at 0x...>
        vals = []
        curr = self
        while curr:
            vals.append(str(curr.val))
            curr = curr.next
        return " -> ".join(vals) + " -> None"

# Build a list: 1 → 2 → 3 → None
head = ListNode(1, ListNode(2, ListNode(3)))
print(head)                   # 1 -> 2 -> 3 -> None  (because __repr__ is defined)

# Dummy head pattern — avoids edge case when deleting the head node
def delete_node(head: ListNode, target: int) -> ListNode:
    dummy = ListNode(0)          # sentinel: dummy.next = head
    dummy.next = head
    curr = dummy
    while curr.next:
        if curr.next.val == target:
            curr.next = curr.next.next   # skip the target node
        else:
            curr = curr.next
    return dummy.next            # real head (dummy.next, not dummy itself)

result = delete_node(head, 2)
print(result)                 # 1 -> 3 -> None

# Class vs instance variables
class Counter:
    total = 0                  # class variable — shared across ALL instances
    def __init__(self, name):
        self.name = name       # instance variable — unique to each instance
        Counter.total += 1

c1 = Counter("a")
c2 = Counter("b")
print(f"c1.name={c1.name}  c2.name={c2.name}  total={Counter.total}")
# c1.name=a  c2.name=b  total=2  ← total is shared

# Simplicity and clarity is Gold


<a id='5'></a>

## 5. 🧩 Pattern 2: __lt__ for Heap Ordering — LC 295, 703

---

```
PROBLEM:  heapq uses < to compare elements. Pushing custom objects requires
          __lt__ — otherwise you get TypeError: '<' not supported.

APPROACH A — define __lt__:
  class Task:
      def __init__(self, priority, name):
          self.priority = priority
          self.name = name
      def __lt__(self, other):
          return self.priority < other.priority   # smaller priority = higher in min-heap

APPROACH B — tuple wrapping (simpler, no class change needed):
  heapq.heappush(heap, (priority, task))
  ← tuple comparison: first element compared first

TIE-BREAKING:
  If two priorities are equal, heapq compares the next element.
  If the object doesn't support <, Python raises TypeError.
  Fix: heapq.heappush(heap, (priority, counter, task))
  where counter is a unique incrementing int — guarantees no object comparison needed.

SLOW MOTION TRACE — push Task(3,'C'), Task(1,'A'), Task(2,'B'):
  heap after push(3,'C'): [(3,'C')]
  heap after push(1,'A'): [(1,'A'), (3,'C')]   ← 1<3, sift up
  heap after push(2,'B'): [(1,'A'), (3,'C'), (2,'B')]
  heappop() → Task(1,'A')  ← min priority out first
```


In [ ]:
import heapq
from typing import List

# ── Approach A: __lt__ on the class ──────────────────────────────────────────
class Task:
    def __init__(self, priority: int, name: str):
        self.priority = priority
        self.name = name

    def __lt__(self, other: 'Task') -> bool:
        return self.priority < other.priority   # min-heap by priority

    def __repr__(self) -> str:
        return f"Task({self.priority}, '{self.name}')"

heap = []
for p, n in [(3,'C'), (1,'A'), (2,'B')]:
    heapq.heappush(heap, Task(p, n))

print("Heap (min-priority first):")
while heap:
    print(f"  {heapq.heappop(heap)}")

# ── Approach B: tuple wrapping ────────────────────────────────────────────────
# No class change needed — tuples compare element by element
import itertools
counter = itertools.count()         # unique tiebreaker — prevents object comparison

heap2 = []
for p, n in [(3,'C'), (1,'A'), (1,'D'), (2,'B')]:    # two priority=1 tasks
    heapq.heappush(heap2, (p, next(counter), n))      # (priority, tie, name)

print("\nTuple heap (priority then insertion order):")
while heap2:
    p, _, n = heapq.heappop(heap2)
    print(f"  priority={p}  name={n}")

# ── LC 703 — KthLargest (min-heap of size k) ──────────────────────────────────
class KthLargest:
    """
    LC 703 — Kth Largest Element in a Stream.
    Approach: maintain min-heap of size k. Root is always the kth largest.
    Time: O(n log k) init, O(log k) per add.
    Space: O(k) — heap holds at most k elements.
    """
    def __init__(self, k: int, nums: List[int]):
        self.k    = k
        self.heap = []                          # min-heap of size k
        for n in nums:
            self.add(n)

    def add(self, val: int) -> int:
        heapq.heappush(self.heap, val)          # push new value
        while len(self.heap) > self.k:
            heapq.heappop(self.heap)            # evict smallest — keep k largest
        return self.heap[0]                     # root = kth largest

kth = KthLargest(3, [4, 5, 8, 2])
print("\nKthLargest(k=3):")
for val in [3, 5, 10, 9, 4]:
    print(f"  add({val}) → {kth.add(val)}")   # 4, 5, 8, 8, 8

# Simplicity and clarity is Gold


<a id='6'></a>

## 6. 🧩 Pattern 3: __iter__ / __next__ — Iterator Protocol — LC 173, 341

---

```
PROBLEM:  Make a custom object iterable — usable in for loops, list(), next() calls.

ITERATOR PROTOCOL:
  __iter__: called once at start of for loop. Must return an iterator object.
  __next__: called once per loop step. Returns next item or raises StopIteration.

  for x in obj:     →   it = iter(obj)       # calls obj.__iter__()
                        while True:
                            try: x = next(it)  # calls it.__next__()
                            except StopIteration: break
                            ... loop body ...

ITERABLE vs ITERATOR:
  Iterable: has __iter__ (list, dict, range, custom class)
  Iterator: has both __iter__ and __next__ (can be advanced with next())
  An iterator is also an iterable (its __iter__ returns self).

TWO PATTERNS:
  1. Class is both iterable and iterator — __iter__ returns self, __next__ advances
  2. Class is iterable — __iter__ returns a SEPARATE iterator object (cleaner, reusable)

SLOW MOTION on BSTIterator (LC 173):
  BST: [7, 3, 15, null, null, 9, 20]
  In-order: 3, 7, 9, 15, 20
  __init__: flatten tree to sorted list [3,7,9,15,20], index=0
  next(): return list[index]; index++
  hasNext(): return index < len(list)
```


In [ ]:
from typing import Optional, List

# ── Simple iterator class — Range reimplemented ───────────────────────────────
class MyRange:
    """Like range(start, stop) but implemented with __iter__ / __next__."""
    def __init__(self, start: int, stop: int):
        self.current = start
        self.stop    = stop

    def __iter__(self):
        return self            # iterator is self — common pattern for simple iterators

    def __next__(self) -> int:
        if self.current >= self.stop:
            raise StopIteration   # signals for loop to end — REQUIRED
        val = self.current
        self.current += 1
        return val

print("MyRange(1, 5):")
for x in MyRange(1, 5):
    print(f"  {x}", end="")
print()

# list() and sum() consume iterators too
print(list(MyRange(0, 4)))    # [0,1,2,3]
print(sum(MyRange(1, 6)))     # 15

# Reusable iterable — __iter__ returns a NEW object each time
class Squares:
    """Reusable iterable: each for loop gets a fresh iterator."""
    def __init__(self, n: int):
        self.n = n

    def __iter__(self):
        return SquaresIterator(self.n)    # fresh iterator, not self

class SquaresIterator:
    def __init__(self, n):
        self.i = 1
        self.n = n
    def __iter__(self): return self
    def __next__(self):
        if self.i > self.n: raise StopIteration
        val = self.i ** 2
        self.i += 1
        return val

sq = Squares(4)
print(list(sq))    # [1, 4, 9, 16]
print(list(sq))    # [1, 4, 9, 16] — fresh iterator, works again!

# ── LC 173 — BST Iterator ─────────────────────────────────────────────────────
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val; self.left = left; self.right = right

class BSTIterator:
    """
    LC 173 — Binary Search Tree Iterator.
    Approach: flatten to sorted list in __init__; iterate with index.
    Time: O(n) init, O(1) next/hasNext.  Space: O(n) for sorted list.
    """
    def __init__(self, root: Optional[TreeNode]):
        self.nodes: List[int] = []
        self._inorder(root)
        self.index = 0

    def _inorder(self, node):
        if not node: return
        self._inorder(node.left)
        self.nodes.append(node.val)
        self._inorder(node.right)

    def next(self) -> int:
        val = self.nodes[self.index]
        self.index += 1
        return val

    def hasNext(self) -> bool:
        return self.index < len(self.nodes)

    def __iter__(self): return self

    def __next__(self) -> int:
        if not self.hasNext(): raise StopIteration
        return self.next()

# Build BST: 7 with children 3, 15(9,20)
root = TreeNode(7, TreeNode(3), TreeNode(15, TreeNode(9), TreeNode(20)))
it = BSTIterator(root)
print("\nBSTIterator in-order:")
while it.hasNext():
    print(f"  {it.next()}", end="")
print()

# Simplicity and clarity is Gold


<a id='7'></a>

## 7. 🧩 Pattern 4: @property, @staticmethod, @classmethod

---

```
@property       — getter: access method like an attribute (no parentheses)
@<name>.setter  — setter: assign to a "method" like an attribute
@staticmethod   — utility function that belongs to the class but needs no instance/class
@classmethod    — factory method that receives the CLASS as first arg (cls, not self)

WHEN TO USE:
  @property:      computed attribute (e.g. full_name from first + last)
                  validation on assignment (e.g. prevent negative age)
  @staticmethod:  helper function logically grouped with the class
                  doesn't touch self or cls
  @classmethod:   alternative constructors (factory pattern)
                  subclass-aware construction

SLOW MOTION:
  class Circle:
      def __init__(self, radius): self._radius = radius

      @property
      def radius(self): return self._radius              ← c.radius (no ())

      @radius.setter
      def radius(self, val):
          if val < 0: raise ValueError
          self._radius = val                             ← c.radius = 5

      @staticmethod
      def unit():
          return Circle(1)                               ← Circle.unit()

      @classmethod
      def from_diameter(cls, d):
          return cls(d / 2)                              ← Circle.from_diameter(10)
```


In [ ]:
import math

class Circle:
    def __init__(self, radius: float):
        self._radius = radius      # convention: _name means "private" (not enforced by Python)

    @property
    def radius(self) -> float:
        return self._radius        # accessed as c.radius, not c.radius()

    @radius.setter
    def radius(self, val: float) -> None:
        if val < 0:
            raise ValueError(f"radius must be non-negative, got {val}")
        self._radius = val         # accessed as c.radius = 5

    @property
    def area(self) -> float:
        return math.pi * self._radius ** 2   # computed on demand, read-only

    @staticmethod
    def unit() -> 'Circle':
        """Factory for a unit circle — no instance needed, pure utility."""
        return Circle(1)           # Circle.unit() — no self, no cls

    @classmethod
    def from_diameter(cls, diameter: float) -> 'Circle':
        """Alternative constructor — cls-aware so subclasses work too."""
        return cls(diameter / 2)   # Circle.from_diameter(10)

    def __repr__(self) -> str:
        return f"Circle(r={self._radius:.2f})"

c = Circle(5)
print(c.radius)           # 5  — @property, no parentheses
print(f"{c.area:.2f}")    # 78.54

c.radius = 10             # @setter, assignment syntax
print(c)                  # Circle(r=10.00)

try:
    c.radius = -1         # triggers validation
except ValueError as e:
    print(f"Caught: {e}")

unit = Circle.unit()
print(unit)               # Circle(r=1.00)

from_diam = Circle.from_diameter(20)
print(from_diam)          # Circle(r=10.00)

# ── LC context: @property on a node ──────────────────────────────────────────
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    @property
    def is_leaf(self) -> bool:
        return self.left is None and self.right is None  # computed, not stored

leaf = TreeNode(5)
print(f"leaf.is_leaf = {leaf.is_leaf}")   # True

# Simplicity and clarity is Gold


<a id='8'></a>

## 8. 🧩 Pattern 5: MinStack — LC 155

---

```
PROBLEM:  Design a stack that supports push, pop, top, and getMin in O(1).

APPROACH: Maintain a second "min stack" in parallel.
          min_stack[i] = minimum of all elements at stack depth 0..i.
          When you push x, push min(x, current_min) to min_stack.
          When you pop, pop from both stacks simultaneously.

SLOW MOTION TRACE: push(5), push(3), push(7), push(2), pop():
  step         stack          min_stack     getMin()
  push(5)      [5]            [5]           5
  push(3)      [5,3]          [5,3]         3     (3 < 5, new min)
  push(7)      [5,3,7]        [5,3,3]       3     (7 >= 3, keep 3)
  push(2)      [5,3,7,2]      [5,3,3,2]     2
  pop()        [5,3,7]        [5,3,3]       3     (2 gone, min reverts)
  getMin()  → 3  ✓

KEY INSIGHT: min_stack[i] carries the minimum SEEN so far, not just the current level.
             That's why getMin() is O(1) — you don't need to scan.

TIME / SPACE:
  push/pop/top/getMin: O(1)
  Space: O(n) — two parallel stacks
```


In [ ]:
class MinStack:
    """
    LC 155 — Min Stack.
    Approach: parallel min_stack tracks running minimum at each depth.
    Time: O(1) all operations.  Space: O(n).
    """
    def __init__(self):
        self.stack     = []    # main storage
        self.min_stack = []    # min_stack[i] = min of stack[0..i]

    def push(self, val: int) -> None:
        self.stack.append(val)
        # running min: either new val is smaller, or current min stays
        current_min = val if not self.min_stack else min(val, self.min_stack[-1])
        self.min_stack.append(current_min)

    def pop(self) -> None:
        self.stack.pop()
        self.min_stack.pop()       # must pop both — they stay in sync

    def top(self) -> int:
        return self.stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]  # O(1) — top of min_stack is current minimum

    def __repr__(self) -> str:
        return f"MinStack(stack={self.stack}, mins={self.min_stack})"

# Slow motion trace
s = MinStack()
for val in [5, 3, 7, 2]:
    s.push(val)
    print(f"  push({val:2d}) → stack={s.stack}  mins={s.min_stack}  getMin={s.getMin()}")

s.pop()
print(f"  pop()    → stack={s.stack}  mins={s.min_stack}  getMin={s.getMin()}")

# Test harness
def test_minstack():
    s = MinStack()
    s.push(-2); s.push(0); s.push(-3)
    assert s.getMin() == -3, f"expected -3, got {s.getMin()}"
    s.pop()
    assert s.top() == 0,     f"expected 0, got {s.top()}"
    assert s.getMin() == -2, f"expected -2, got {s.getMin()}"
    print("MinStack: 3/3 tests passed")

test_minstack()
print("MinStack defined.")

# Simplicity and clarity is Gold


<a id='9'></a>

## 9. 🧩 Pattern 6: MedianFinder — Two-Heap Design — LC 295

---

```
PROBLEM:  Design a data structure that supports addNum(int) and findMedian().
          findMedian() must run in O(1).

APPROACH: Split the stream into two halves using two heaps:
  max_heap (left half)  — stores smaller half, peeking gives largest of small
  min_heap (right half) — stores larger half, peeking gives smallest of large

  INVARIANT:
    len(max_heap) == len(min_heap)        ← even count: median = avg of two tops
    OR len(max_heap) == len(min_heap) + 1 ← odd count: median = max_heap top

  REBALANCE after each insert:
    1. Always push to max_heap first (negate for Python's min-heap)
    2. If max_heap top > min_heap top: move max_heap top to min_heap
    3. If sizes differ by 2: balance by moving one element

SLOW MOTION on stream [1, 2, 3]:
  add(1): max=[-1]        min=[]     sizes(1,0)  → med=1
  add(2): max=[-1]        min=[2]    sizes(1,1)  → med=(1+2)/2=1.5
  add(3): max=[-2,-1]     min=[3]    sizes(2,1)  → med=2

TIME / SPACE:
  addNum:    O(log n) — heap push/pop
  findMedian: O(1)   — peek tops
  Space:      O(n)   — two heaps
```


In [ ]:
import heapq

class MedianFinder:
    """
    LC 295 — Find Median from Data Stream.
    Approach: max_heap (left half) + min_heap (right half). Rebalance after each add.
    Time: O(log n) addNum, O(1) findMedian.  Space: O(n).
    """
    def __init__(self):
        self.max_heap = []   # left half — Python min-heap, store negated values
        self.min_heap = []   # right half — standard min-heap

    def addNum(self, num: int) -> None:
        # Step 1: push to max_heap (negate for max semantics in Python's min-heap)
        heapq.heappush(self.max_heap, -num)

        # Step 2: ensure max_heap top <= min_heap top (heaps don't overlap)
        if self.min_heap and (-self.max_heap[0]) > self.min_heap[0]:
            val = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, val)

        # Step 3: rebalance size — max_heap may have at most 1 extra element
        if len(self.max_heap) > len(self.min_heap) + 1:
            val = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, val)
        elif len(self.min_heap) > len(self.max_heap):
            val = heapq.heappop(self.min_heap)
            heapq.heappush(self.max_heap, -val)

    def findMedian(self) -> float:
        if len(self.max_heap) > len(self.min_heap):
            return float(-self.max_heap[0])   # odd count: max_heap has the middle
        return (-self.max_heap[0] + self.min_heap[0]) / 2.0  # even: avg of two middles

# Slow motion trace
mf = MedianFinder()
for num in [1, 2, 3, 4, 5]:
    mf.addNum(num)
    print(f"  add({num}) → max_heap={[-x for x in sorted(mf.max_heap)]}  min_heap={sorted(mf.min_heap)}  median={mf.findMedian()}")

# Test harness
def test_median():
    mf = MedianFinder()
    mf.addNum(1); mf.addNum(2)
    assert mf.findMedian() == 1.5, f"expected 1.5 got {mf.findMedian()}"
    mf.addNum(3)
    assert mf.findMedian() == 2.0, f"expected 2.0 got {mf.findMedian()}"
    mf.addNum(-1)
    assert mf.findMedian() == 1.5, f"expected 1.5 got {mf.findMedian()}"
    print("MedianFinder: 4/4 tests passed")

test_median()
print("MedianFinder defined.")

# Simplicity and clarity is Gold


<a id='10'></a>

## 10. 🧩 Pattern 7: LRUCache Skeleton — LC 146

---

```
PROBLEM:  Design LRU (Least Recently Used) cache with capacity.
          get(key): return value or -1, mark as most recently used.
          put(key, val): insert or update, evict LRU if over capacity.

APPROACH: OrderedDict — O(1) insertion, deletion, and move-to-end.
          Most recently used → end. Least recently used → front.

SLOW MOTION on capacity=2: put(1,1), put(2,2), get(1), put(3,3), get(2):
  put(1,1): {1:1}            1 is MRU
  put(2,2): {1:1, 2:2}       2 is MRU
  get(1):   {2:2, 1:1}       moved 1 to end (MRU)  → return 1
  put(3,3): evict LRU=2 (front), add 3 → {1:1, 3:3}
  get(2):   2 not found → -1

KEY INSIGHT: OrderedDict.move_to_end(key) + popitem(last=False) = O(1) LRU.
             Without OrderedDict: use doubly linked list + hashmap manually.

TIME / SPACE:
  get/put: O(1)  Space: O(capacity)
```


In [ ]:
from collections import OrderedDict

class LRUCache:
    """
    LC 146 — LRU Cache.
    Approach: OrderedDict — insertion order = recency order.
    MRU at end (last=True), LRU at front (last=False).
    Time: O(1) get/put.  Space: O(capacity).
    """
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()     # key → value, ordered by recency

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)    # mark as most recently used (move to end)
        return self.cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)    # update recency
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # evict LRU (front of ordered dict)

    def __repr__(self) -> str:
        return f"LRUCache({dict(self.cache)})  LRU←→MRU"

# Slow motion trace
lru = LRUCache(2)
lru.put(1, 1); print(f"put(1,1): {lru}")
lru.put(2, 2); print(f"put(2,2): {lru}")
print(f"get(1)={lru.get(1)}:  {lru}")       # returns 1, 1 becomes MRU
lru.put(3, 3); print(f"put(3,3): {lru}")    # 2 evicted (LRU)
print(f"get(2)={lru.get(2)}")               # -1 (evicted)
print(f"get(3)={lru.get(3)}")               # 3
lru.put(4, 4); print(f"put(4,4): {lru}")    # 1 evicted
print(f"get(1)={lru.get(1)}")               # -1 (evicted)
print(f"get(3)={lru.get(3)}")               # 3
print(f"get(4)={lru.get(4)}")               # 4

print("LRUCache defined.")

# Simplicity and clarity is Gold


<a id='12'></a>

## 12. Full Decision Map

```
NEED TO                                    DUNDER / DECORATOR     LC
──────────────────────────────────────────────────────────────────────────────
Push custom objects onto heap              __lt__                 295, 703
Iterate object in for loop                 __iter__ + __next__    173, 341
Computed read-only attribute               @property              any class
Validate on assignment                     @property + .setter    any class
Alternative constructor                    @classmethod           any class
Utility function grouped with class        @staticmethod          any class
Print useful info in REPL/debug            __repr__               always
Stack with O(1) minimum                    class + two stacks     155
Median stream                              class + two heaps      295
LRU eviction                               class + OrderedDict    146
Kth largest stream                         class + min-heap       703
BST traversal as iterator                  __iter__ + __next__    173
──────────────────────────────────────────────────────────────────────────────
```


<a id='13'></a>

## 13. Interview Cheat Sheet

### 1. When to reach for OOP:

| Signal | What to Do |
|--------|------------|
| "design a data structure" | Stateful class with __init__ |
| "push custom objects to heap" | Define __lt__ or use tuple wrapper |
| "make object iterable" | __iter__ returning self + __next__ |
| "computed attribute" | @property |
| "alternative constructor" | @classmethod |
| "debug output" | __repr__ |

### 2. Templates to memorize:

```python
# HEAP WITH CUSTOM OBJECT — always safe approach
import heapq, itertools
counter = itertools.count()
heap = []
heapq.heappush(heap, (priority, next(counter), obj))

# ITERATOR PROTOCOL — minimal
class MyIter:
    def __init__(self, data): self.data = data; self.i = 0
    def __iter__(self): return self
    def __next__(self):
        if self.i >= len(self.data): raise StopIteration
        val = self.data[self.i]; self.i += 1; return val

# STATEFUL CLASS SKELETON
class MyStructure:
    def __init__(self, capacity):
        self.capacity = capacity
        self._storage = []      # or dict, heap, deque

    def operation(self, val):
        ...
```

### 3. Gotchas:

```
❌  Push objects to heap without __lt__ → TypeError at runtime
❌  __iter__ returns self but forgot to reset index → iterator can't restart
❌  Mutable default: def __init__(self, data=[]) → shared across instances
✅  tuple(priority, counter, obj) — safest heap approach, no __lt__ needed
✅  __repr__ always worth defining — saves you from <__main__.X object at 0x...>
✅  MedianFinder: negate values for max_heap (Python only has min-heap)
✅  LRU: move_to_end(key) + popitem(last=False) = O(1) with OrderedDict
```


```
OOP & CLASS PATTERNS MASTER MAP
════════════════════════════════════════════════════

             CUSTOM CLASS
                  │
    ┌─────────────┼──────────────┐
    ▼             ▼              ▼
 DUNDER        DECORATORS    STATEFUL
 METHODS                     DESIGNS
    │             │              │
 __init__     @property      MinStack
 __repr__     @setter          155
 __lt__       @classmethod   MedianFinder
  (heap)      @staticmethod    295
 __iter__                    LRUCache
 __next__                      146
  (loop)                    KthLargest
 __enter__                     703
 __exit__
  (with)
    │
    ▼
HEAP ORDERING:
  __lt__ on class
  OR (priority, counter, obj)
  — always pick tuple for safety
```

---
*End of OOP & Class Patterns Master Guide — Sean Edition*
